# Nano-vLLM on Google Colab

Run [nano-vllm](https://github.com/GeeeekExplorer/nano-vllm) on a Colab GPU with Qwen3-0.6B.

**Before you start:** set the runtime to a GPU. *Runtime → Change runtime type → T4 GPU* (free tier) or A100/L4 (Pro). Tensor parallelism is disabled here since Colab gives you one GPU.

**Expected timings (T4):** ~5 min for installs (mostly the flash-attn wheel), ~1 min model download, then generation is instant.

## 1. Sanity-check the GPU

If `nvidia-smi` fails, the runtime isn't on a GPU — fix that before continuing.

In [ ]:
!nvidia-smi

## 2. Install flash-attn (the tricky one)

Building from source on Colab takes ~30 min. Instead, we detect the torch / CUDA / Python combo and grab a prebuilt wheel from the [Dao-AILab releases](https://github.com/Dao-AILab/flash-attention/releases). If no matching wheel exists, this cell falls back to a slow source build.

In [ ]:
import sys, re, subprocess, urllib.request, json, torch

py = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = ".".join(torch.__version__.split(".")[:2])  # e.g. "2.5"
cuda_ver = torch.version.cuda.replace(".", "")[:3] if torch.version.cuda else ""  # e.g. "124" -> "12"
cuda_major = cuda_ver[:2]  # "12"
print(f"python={py} torch={torch_ver} cuda={cuda_major}")

# Query the GitHub releases API for the newest matching wheel.
with urllib.request.urlopen("https://api.github.com/repos/Dao-AILab/flash-attention/releases?per_page=5") as r:
    releases = json.load(r)

pattern = re.compile(
    rf"flash_attn-[\d.]+(?:\.post\d+)?\+cu{cuda_major}\d?torch{torch_ver}cxx11abiFALSE-{py}-{py}-linux_x86_64\.whl$"
)
wheel_url = None
for rel in releases:
    for asset in rel.get("assets", []):
        if pattern.search(asset["name"]):
            wheel_url = asset["browser_download_url"]
            break
    if wheel_url:
        break

if wheel_url:
    print(f"Installing prebuilt wheel: {wheel_url.rsplit('/', 1)[-1]}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", wheel_url])
else:
    print("No matching prebuilt wheel found — falling back to source build (~30 min).")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"])

## 3. Install nano-vllm and its other deps

`triton`, `transformers`, and `xxhash` are required by nano-vllm. `huggingface_hub[cli]` gives us the `hf` command for the model download.

In [ ]:
!pip install -q "triton>=3.0.0" "transformers>=4.51.0" xxhash "huggingface_hub[cli]"
!pip install -q git+https://github.com/GeeeekExplorer/nano-vllm.git

## 4. Download Qwen3-0.6B

~1.2 GB. Goes to `/content/Qwen3-0.6B/`.

In [ ]:
!hf download Qwen/Qwen3-0.6B --local-dir /content/Qwen3-0.6B

## 5. Run the example

Mirrors `example.py` from the repo. `enforce_eager=True` skips CUDA-graph capture so startup is fast; flip to `False` for the bench.

In [ ]:
from nanovllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = "/content/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(MODEL, enforce_eager=True, tensor_parallel_size=1)

prompts = [
    "Introduce yourself in one sentence.",
    "List the first ten prime numbers.",
]
chat_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for p in prompts
]

sampling_params = SamplingParams(temperature=0.6, max_tokens=256)
outputs = llm.generate(chat_prompts, sampling_params)

for p, out in zip(prompts, outputs):
    print(f"\n>>> {p}\n{out['text']}")

## 6. Throughput benchmark (optional)

A scaled-down version of `bench.py` — 32 random sequences instead of 256 so it fits the free-tier T4 comfortably. Bump `num_seqs` if you're on an A100/L4.

First run captures CUDA graphs (~30 s); subsequent calls are fast.

In [ ]:
import time
from random import randint, seed
from nanovllm import LLM, SamplingParams

# Reinitialize without enforce_eager so CUDA graphs kick in.
del llm
import gc, torch
gc.collect(); torch.cuda.empty_cache()

llm = LLM(MODEL, enforce_eager=False, max_model_len=4096, tensor_parallel_size=1)

seed(0)
num_seqs = 32
prompt_token_ids = [
    [randint(0, 10000) for _ in range(randint(100, 512))]
    for _ in range(num_seqs)
]
sampling_params = [
    SamplingParams(temperature=0.6, ignore_eos=True, max_tokens=randint(100, 512))
    for _ in range(num_seqs)
]

# Warm up.
llm.generate(["warmup"], SamplingParams(max_tokens=8), use_tqdm=False)

t = time.time()
llm.generate(prompt_token_ids, sampling_params, use_tqdm=False)
elapsed = time.time() - t

total = sum(sp.max_tokens for sp in sampling_params)
print(f"{num_seqs} seqs, {total} tokens, {elapsed:.2f}s, {total/elapsed:.1f} tok/s")